# Train the exam-score model

This notebook uses four inputs (study hours, attendance, mental-health rating, and sleep hours) to predict `exam_scores`. Part-time-job status is not used, and the score is never included as an input. The source workbook stays local and is not embedded in this notebook.

Set `SIS_DATASET_PATH` to the approved CSV or Excel file, or use the default OneDrive workbook path below.


In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "scripts" / "retrain_model.py").is_file():
    project_root = project_root.parent
if not (project_root / "scripts" / "retrain_model.py").is_file():
    raise FileNotFoundError("Run this notebook from the project or models directory.")
sys.path.insert(0, str(project_root))

dataset_path = Path(
    os.environ.get(
        "SIS_DATASET_PATH",
        str(Path.home() / "OneDrive" / "student_habits_performance.xlsx"),
    )
)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Training dataset not found: {dataset_path}")
from scripts.retrain_model import train_model


In [ ]:
metrics = train_model(str(dataset_path))
print(f"Training rows: {metrics['rows']}")
print(f"Incomplete rows skipped: {metrics['dropped_rows']}")
print(f"Holdout R-squared: {metrics['r2']:.4f}")
print(f"Holdout MAE: {metrics['mae']:.2f} points")
print(f"Holdout RMSE: {metrics['rmse']:.2f} points")


In [ ]:
import joblib
import pandas as pd

model_path = project_root / "models" / "best_model.pkl"
estimator = joblib.load(model_path)
expected_features = [
    "study_hours",
    "attendance",
    "mental_health",
    "sleep_hours",
]
if list(estimator.feature_names_in_) != expected_features:
    raise ValueError(f"Unexpected model features: {list(estimator.feature_names_in_)}")
sample = pd.DataFrame([{
    "study_hours": 3.0,
    "attendance": 85.0,
    "mental_health": 7.0,
    "sleep_hours": 7.0,
}])
sample_score = float(estimator.predict(sample)[0])
print(f"Model inputs verified: {', '.join(expected_features)}")
print(f"Sample prediction: {sample_score:.1f}")
